In [1]:
from pathlib import Path

print("Working directory:", Path.cwd())

Working directory: D:\college\sem6\aml\Week - 8 - Getting Started with NLP - Text Pre - processing and Text Representations


## Helper Function for Text Cleaning:

Implement a Helper Function as per Text Preprocessing Notebook and Complete the following pipeline.

# Build a Text Cleaning Pipeline

In [2]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer, PorterStemmer

nltk.download('stopwords')
nltk.download('wordnet')

stop_words = set(stopwords.words('english'))

def lower_order(text):
  return text.lower()

def remove_urls(text):
  url_pattern = re.compile(r'https?://\S+|www\.\S+')
  return url_pattern.sub(r'', text)

def remove_emoji(string):
  emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"
                           u"\U0001F300-\U0001F5FF"
                           u"\U0001F680-\U0001F6FF"
                           u"\U0001F1E0-\U0001F1FF"
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
  return emoji_pattern.sub(r' ', string)

def removeunwanted_characters(document):
  document = re.sub("@[A-Za-z0-9_]+", " ", document)
  document = re.sub("#[A-Za-z0-9_]+", "", document)
  document = re.sub("[^0-9A-Za-z ]", "", document)
  document = remove_emoji(document)
  document = document.replace('  ', ' ')
  return document.strip()

def remove_stopwords(text_tokens):
  return [token for token in text_tokens if token not in stop_words]

def lemmatization(token_text):
  wordnet = WordNetLemmatizer()
  return [wordnet.lemmatize(token, pos='v') for token in token_text]

def stemming(text):
  porter = PorterStemmer()
  return [porter.stem(word) for word in text]

def text_cleaning_pipeline(dataset, rule="lemmatize"):
  """
  Compile a complete text-cleaning pipeline.
  """
  if not isinstance(dataset, str):
    dataset = str(dataset)

  # Convert the input to small/lower order.
  data = lower_order(dataset)
  # Remove URLs
  data = remove_urls(data)
  # Remove emojis
  data = remove_emoji(data)
  # Remove all other unwanted characters.
  data = removeunwanted_characters(data)
  # Create tokens.
  tokens = data.split()
  # Remove stopwords
  tokens = remove_stopwords(tokens)

  if rule == "lemmatize":
    tokens = lemmatization(tokens)
  elif rule == "stem":
    tokens = stemming(tokens)
  else:
    print("Pick between lemmatize or stem")

  return " ".join(tokens)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Lenovo\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


# Text Classification using Machine Learning Models


### 📝 Instructions: Trump Tweet Sentiment Classification

1. **Load the Dataset**  
   Load the dataset named `"trump_tweet_sentiment_analysis.csv"` using `pandas`. Ensure the dataset contains at least two columns: `"text"` and `"label"`.

2. **Text Cleaning and Tokenization**  
   Apply a text preprocessing pipeline to the `"text"` column. This should include:
   - Lowercasing the text  
   - Removing URLs, mentions, punctuation, and special characters  
   - Removing stopwords  
   - Tokenization (optional: stemming or lemmatization)
   - "Complete the above function"

3. **Train-Test Split**  
   Split the cleaned and tokenized dataset into **training** and **testing** sets using `train_test_split` from `sklearn.model_selection`.

4. **TF-IDF Vectorization**  
   Import and use the `TfidfVectorizer` from `sklearn.feature_extraction.text` to transform the training and testing texts into numerical feature vectors.

5. **Model Training and Evaluation**  
   Import **Logistic Regression** (or any machine learning model of your choice) from `sklearn.linear_model`. Train it on the TF-IDF-embedded training data, then evaluate it using the test set.  
   - Print the **classification report** using `classification_report` from `sklearn.metrics`.


## 1) Load and Prepare Dataset

In [3]:
import pandas as pd

data = pd.read_csv("trum_tweet_sentiment_analysis.csv", encoding="ISO-8859-1")
data = data.rename(columns={"Sentiment": "label"})
data = data[["text", "label"]].dropna()

print("Shape:", data.shape)
print(data["label"].value_counts())
data.head()

Shape: (1850123, 2)
label
0    1244211
1     605912
Name: count, dtype: int64


,text,label
0,RT @JohnLeguizamo: #trump not draining swamp b...,0
1,ICYMI: Hackers Rig FM Radio Stations To Play A...,0
2,Trump protests: LGBTQ rally in New York https:...,1
3,"""Hi I'm Piers Morgan. David Beckham is awful b...",0
4,RT @GlennFranco68: Tech Firm Suing BuzzFeed fo...,0


## 2) Clean Text and Build Features

In [4]:
# Optional down-sampling for faster local training (keeps class balance)
max_per_class = 20000
sampled_groups = []
for _, grp in data.groupby("label"):
    sampled_groups.append(grp.sample(n=min(max_per_class, len(grp)), random_state=42))

sampled_data = pd.concat(sampled_groups).sample(frac=1, random_state=42).reset_index(drop=True)

sampled_data["clean_text"] = sampled_data["text"].apply(
    lambda x: text_cleaning_pipeline(x, rule="lemmatize")
)

sampled_data = sampled_data[sampled_data["clean_text"].str.len() > 0]

print("Sampled shape:", sampled_data.shape)
sampled_data[["text", "clean_text", "label"]].head()

Sampled shape: (39995, 3)


,text,clean_text,label
0,RT @MeetThePress: Stephen Miller on Trump's vi...,rt stephen miller trump view general flynn tel...,1
1,Democrats Want Answers on Trump Officials' Tie...,democrats want answer trump officials tie russia,0
2,RT @williamlegate: @realDonaldTrump @ABC \r\r\...,rt alsoi second read like child sad,1
3,RT @CertifiedFool_: IM TIRED OF HEARING ABOUT ...,rt im tire hear donald trump fuck ass administ...,0
4,@beardedcrank @RoseAnnDeMoro @NomikiKonst @Huf...,dont think demoro reason 10k voters pawimi vot...,1


## 3) Train and Evaluate ML Models

In [5]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

X_train, X_test, y_train, y_test = train_test_split(
    sampled_data["clean_text"],
    sampled_data["label"],
    test_size=0.2,
    random_state=42,
    stratify=sampled_data["label"]
)

vectorizer = TfidfVectorizer(max_features=30000, ngram_range=(1, 2))
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

models = {
    "LogisticRegression": LogisticRegression(max_iter=1000),
    "LinearSVC": LinearSVC()
}

for name, model in models.items():
    model.fit(X_train_vec, y_train)
    preds = model.predict(X_test_vec)

    print(f"\n===== {name} =====")
    print("Accuracy:", round(accuracy_score(y_test, preds), 4))
    print(classification_report(y_test, preds))


===== LogisticRegression =====
Accuracy: 0.8671
              precision    recall  f1-score   support

           0       0.86      0.88      0.87      3999
           1       0.87      0.86      0.87      4000

    accuracy                           0.87      7999
   macro avg       0.87      0.87      0.87      7999
weighted avg       0.87      0.87      0.87      7999




===== LinearSVC =====
Accuracy: 0.8921
              precision    recall  f1-score   support

           0       0.89      0.89      0.89      3999
           1       0.89      0.89      0.89      4000

    accuracy                           0.89      7999
   macro avg       0.89      0.89      0.89      7999
weighted avg       0.89      0.89      0.89      7999



In [6]:
test_text = "The policy speech was full of energy and optimism!"
clean_test_text = text_cleaning_pipeline(test_text)

best_model = models["LinearSVC"]
pred_label = best_model.predict(vectorizer.transform([clean_test_text]))[0]

print("Original:", test_text)
print("Cleaned:", clean_test_text)
print("Predicted label:", pred_label)

Original: The policy speech was full of energy and optimism!
Cleaned: policy speech full energy optimism
Predicted label: 1
